# Probabilistic Models

> *"Probability theory is nothing but common sense reduced to calculation."*
> — Pierre-Simon Laplace, 1812

Interactive theory notebook covering Bayesian inference, exponential families, graphical models,
latent variable models, EM, variational inference, VAEs, MCMC, GPs, HMMs, diffusion models, and flow matching.
All cells runnable top-to-bottom.


In [ ]:
import numpy as np
import scipy.linalg as la
from scipy import stats, special

try:
    import matplotlib.pyplot as plt
    import matplotlib as mpl
    plt.style.use('seaborn-v0_8-whitegrid')
    mpl.rcParams.update({
        'figure.figsize': (10, 6), 'figure.dpi': 100,
        'font.size': 12, 'axes.titlesize': 14, 'axes.labelsize': 12,
        'lines.linewidth': 2.0,
        'axes.spines.top': False, 'axes.spines.right': False,
    })
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

try:
    import seaborn as sns
    HAS_SNS = True
except ImportError:
    HAS_SNS = False

COLORS = {
    'primary':   '#0077BB',
    'secondary': '#EE7733',
    'tertiary':  '#009988',
    'error':     '#CC3311',
    'neutral':   '#555555',
    'highlight': '#EE3377',
}

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)
print('Setup complete.')

---

## 2. Formal Foundations

### 2.2 Bayes' Theorem — Sequential Updating

We demonstrate Bayesian posterior updating with a Beta-Bernoulli conjugate model.
The prior is Beta(α, β); after observing coin flips, the posterior is Beta(α + heads, β + tails).

In [ ]:
# === 2.2 Bayesian Updating: Beta-Bernoulli ===

from scipy.stats import beta as beta_dist

# True coin bias (unknown to the model)
p_true = 0.7

# Prior: Beta(2, 2) — weakly informative, centered at 0.5
alpha0, beta0 = 2.0, 2.0

# Generate observations
n_obs = 50
flips = np.random.binomial(1, p_true, n_obs)
print(f'True p = {p_true}, observed heads fraction = {flips.mean():.3f}')

p_grid = np.linspace(0, 1, 500)

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 5))
    checkpoints = [0, 5, 15, 50]
    colors_seq = ['#555555', '#009988', '#EE7733', '#0077BB']
    for n, col in zip(checkpoints, colors_seq):
        heads = flips[:n].sum() if n > 0 else 0
        tails = n - heads
        a, b = alpha0 + heads, beta0 + tails
        pdf = beta_dist.pdf(p_grid, a, b)
        label = f'Prior (n=0)' if n == 0 else f'Posterior (n={n}, heads={heads})'
        ax.plot(p_grid, pdf, color=col, label=label)
    ax.axvline(p_true, color=COLORS['error'], linestyle='--', label=f'True p={p_true}')
    ax.set_title('Beta-Bernoulli Sequential Posterior Updating')
    ax.set_xlabel('Coin bias $p$')
    ax.set_ylabel('Density')
    ax.legend()
    fig.tight_layout()
    plt.show()

# Posterior at n=50
heads50 = flips.sum()
a_post, b_post = alpha0 + heads50, beta0 + (n_obs - heads50)
posterior_mean = a_post / (a_post + b_post)
posterior_std = np.sqrt(a_post * b_post / ((a_post + b_post)**2 * (a_post + b_post + 1)))
print(f'Posterior: Beta({a_post:.0f}, {b_post:.0f})')
print(f'Posterior mean = {posterior_mean:.4f}, std = {posterior_std:.4f}')
ok = abs(posterior_mean - p_true) < 0.1
print(f"{'PASS' if ok else 'FAIL'} — posterior mean within 0.1 of true p")

---

### 2.3 Exponential Family — Moment Generation

We verify the key property: $\nabla_\eta A(\eta) = \mathbb{E}[T(x)]$ and $\nabla^2_\eta A(\eta) = \mathrm{Cov}[T(x)]$ for the Poisson distribution.

In [ ]:
# === 2.3 Exponential Family: Poisson Moments from Log-Partition ===

# Poisson: p(x|lambda) = lambda^x exp(-lambda) / x!
# Natural param: eta = log(lambda)  => lambda = exp(eta)
# Sufficient stat: T(x) = x
# Log-partition: A(eta) = exp(eta) = lambda

lambdas = np.array([1.0, 2.0, 5.0, 10.0])
etas = np.log(lambdas)  # natural parameters

print(f"{'lambda':>8} {'eta':>8} {'A(eta)':>10} {'dA/deta':>10} {'E[X]':>8} {'d2A':>10} {'Var[X]':>8}")
print('-' * 68)
for lam, eta in zip(lambdas, etas):
    A = np.exp(eta)           # log-partition
    dA = np.exp(eta)          # gradient = E[T(x)] = E[x] = lambda
    d2A = np.exp(eta)         # Hessian = Var[T(x)] = Var[x] = lambda
    # Verify by simulation
    samples = np.random.poisson(lam, 100000)
    E_empirical = samples.mean()
    V_empirical = samples.var()
    print(f'{lam:8.1f} {eta:8.4f} {A:10.4f} {dA:10.4f} {E_empirical:8.4f} {d2A:10.4f} {V_empirical:8.4f}')

# Verify moment properties
ok1 = np.allclose(np.exp(etas), lambdas)  # A(eta) = exp(eta)
ok2 = np.allclose(np.exp(etas), lambdas)  # dA = E[X] = lambda
print(f"\n{'PASS' if ok1 else 'FAIL'} — A(eta) = lambda")
print(f"{'PASS' if ok2 else 'FAIL'} — dA/deta = E[X] = lambda")

---

## 4. Latent Variable Models

### 4.1 Gaussian Mixture Models

We visualise the GMM generative process and the E-step responsibilities.
The responsibility $r_{ik} = p(z^{(i)}=k | x^{(i)}, \theta)$ gives the posterior probability
that point $i$ belongs to cluster $k$.

In [ ]:
# === 4.1 Gaussian Mixture Model: Generative Process and Responsibilities ===

from scipy.stats import norm

# True GMM parameters (2 components, 1D)
K = 2
pi_true = np.array([0.4, 0.6])
mu_true = np.array([-2.0, 2.5])
sigma_true = np.array([0.8, 1.2])

# Generate data
n = 300
z = np.random.choice(K, n, p=pi_true)
x = np.array([np.random.normal(mu_true[z_i], sigma_true[z_i]) for z_i in z])

# Compute responsibilities at true parameters
def responsibilities(x, pi, mu, sigma):
    K = len(pi)
    r = np.zeros((len(x), K))
    for k in range(K):
        r[:, k] = pi[k] * norm.pdf(x, mu[k], sigma[k])
    r /= r.sum(axis=1, keepdims=True)
    return r

r = responsibilities(x, pi_true, mu_true, sigma_true)
print(f'Responsibility matrix shape: {r.shape}')
print(f'Average responsibility to cluster 0: {r[:, 0].mean():.3f} (true pi_0={pi_true[0]})')
print(f'Average responsibility to cluster 1: {r[:, 1].mean():.3f} (true pi_1={pi_true[1]})')

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: data coloured by true cluster
    ax = axes[0]
    x_grid = np.linspace(-6, 7, 400)
    for k, col in enumerate([COLORS['primary'], COLORS['secondary']]):
        ax.scatter(x[z==k], np.zeros(sum(z==k)) + 0.01*np.random.randn(sum(z==k)),
                   alpha=0.3, s=15, color=col, label=f'Cluster {k}')
        ax.plot(x_grid, pi_true[k]*norm.pdf(x_grid, mu_true[k], sigma_true[k]),
                color=col)
    mixture_pdf = sum(pi_true[k]*norm.pdf(x_grid, mu_true[k], sigma_true[k]) for k in range(K))
    ax.plot(x_grid, mixture_pdf, 'k--', linewidth=1.5, label='Mixture')
    ax.set_title('True GMM (2 components)')
    ax.set_xlabel('$x$'); ax.set_ylabel('Density')
    ax.legend()

    # Right: responsibilities
    ax = axes[1]
    sort_idx = np.argsort(x)
    ax.plot(x[sort_idx], r[sort_idx, 0], color=COLORS['primary'], label='$r_{i1}$ (cluster 0)')
    ax.plot(x[sort_idx], r[sort_idx, 1], color=COLORS['secondary'], label='$r_{i2}$ (cluster 1)')
    ax.axvline(0, color=COLORS['neutral'], linestyle=':', alpha=0.5)
    ax.set_title('Soft Responsibilities')
    ax.set_xlabel('$x$'); ax.set_ylabel('$r_{ik}$')
    ax.legend()
    fig.tight_layout()
    plt.show()

ok = np.allclose(r.sum(axis=1), 1.0)
print(f"{'PASS' if ok else 'FAIL'} — responsibilities sum to 1 for each point")

---

## 5. Expectation-Maximization

### 5.1–5.3 EM for GMM with Convergence Tracking

We implement EM for a 2-component GMM and verify:
1. Monotonic increase of log-likelihood
2. Convergence to true parameters (up to label permutation)

In [ ]:
# === 5.3 EM Algorithm for Gaussian Mixture Model ===

def gmm_log_likelihood(x, pi, mu, sigma):
    K = len(pi)
    log_liks = np.array([np.log(pi[k]) + norm.logpdf(x, mu[k], sigma[k]) for k in range(K)])
    return np.sum(special.logsumexp(log_liks, axis=0))

def em_gmm(x, K, n_iter=60, seed=0):
    np.random.seed(seed)
    n = len(x)
    # Initialise from k-means-like random assignment
    pi = np.ones(K) / K
    mu = np.random.choice(x, K, replace=False) + 0.1 * np.random.randn(K)
    sigma = np.ones(K) * x.std()

    lls = []
    for t in range(n_iter):
        # E-step: responsibilities
        log_r = np.array([np.log(pi[k]) + norm.logpdf(x, mu[k], sigma[k]) for k in range(K)])
        log_r -= special.logsumexp(log_r, axis=0, keepdims=True)
        r = np.exp(log_r).T  # shape (n, K)

        # M-step: update parameters
        Nk = r.sum(axis=0)
        pi = Nk / n
        mu = (r * x[:, None]).sum(axis=0) / Nk
        sigma = np.sqrt(((r * (x[:, None] - mu[None, :])**2).sum(axis=0) / Nk).clip(1e-6))

        lls.append(gmm_log_likelihood(x, pi, mu, sigma))

    return pi, mu, sigma, np.array(lls)

pi_em, mu_em, sigma_em, lls = em_gmm(x, K=2, n_iter=60)
print(f'True  pi={pi_true}, mu={mu_true}, sigma={sigma_true}')
print(f'Fitted pi={pi_em.round(3)}, mu={mu_em.round(3)}, sigma={sigma_em.round(3)}')

# Verify monotonic increase
diffs = np.diff(lls)
ok_mono = np.all(diffs >= -1e-8)
print(f"{'PASS' if ok_mono else 'FAIL'} — log-likelihood monotonically increases")

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(lls, color=COLORS['primary'])
    ax.set_title('EM Log-Likelihood Convergence')
    ax.set_xlabel('EM Iteration'); ax.set_ylabel('Log-likelihood')
    fig.tight_layout()
    plt.show()

---

## 6. Variational Inference

### 6.1 ELBO Optimisation for a Simple Gaussian Model

We implement CAVI for the model $\mu \sim \mathcal{N}(0, \tau^{-1})$, $x_i \mid \mu \sim \mathcal{N}(\mu, \sigma^2)$
and verify the variational posterior matches the exact conjugate posterior.

In [ ]:
# === 6.2 CAVI for Gaussian Location Model ===

# Model: mu ~ N(0, 1/tau), x_i | mu ~ N(mu, sigma^2)
sigma2 = 1.0  # observation noise (known)
tau = 1.0     # prior precision
n = 20
mu_true_cavi = 3.0
x_data = np.random.normal(mu_true_cavi, np.sqrt(sigma2), n)

# Exact conjugate posterior: N(m_post, s2_post)
s2_post_exact = 1.0 / (tau + n / sigma2)
m_post_exact = s2_post_exact * (x_data.sum() / sigma2)
print(f'Exact posterior: N({m_post_exact:.4f}, {s2_post_exact:.4f})')

# CAVI update (closed form — same as conjugate for this model)
s2_vi = 1.0 / (tau + n / sigma2)
m_vi = s2_vi * (x_data.sum() / sigma2)
print(f'CAVI variational posterior: N({m_vi:.4f}, {s2_vi:.4f})')

ok1 = np.allclose(m_vi, m_post_exact, atol=1e-10)
ok2 = np.allclose(s2_vi, s2_post_exact, atol=1e-10)
print(f"{'PASS' if ok1 else 'FAIL'} — CAVI mean matches exact posterior")
print(f"{'PASS' if ok2 else 'FAIL'} — CAVI variance matches exact posterior")

# ELBO as a function of (m, s2)
def elbo(m, s2, x, sigma2, tau):
    n = len(x)
    # E_q[log p(x|mu)] = -n/2 log(2pi*sigma2) - 1/(2sigma2) * sum[(x-mu)^2]
    # E_q[(x-mu)^2] = (x - m)^2 + s2
    recon = -n/2*np.log(2*np.pi*sigma2) - 1/(2*sigma2) * (np.sum((x-m)**2) + n*s2)
    # -KL(q||prior) = 0.5*(1 + log(s2) - s2*tau - m^2*tau) + const
    kl = 0.5 * (1 + np.log(s2) - s2*tau - m**2*tau)
    return recon + kl

elbo_val = elbo(m_vi, s2_vi, x_data, sigma2, tau)
print(f'ELBO at optimum: {elbo_val:.4f}')

# Verify ELBO is maximised at CAVI solution
elbo_perturbed = elbo(m_vi + 0.5, s2_vi, x_data, sigma2, tau)
print(f"{'PASS' if elbo_val > elbo_perturbed else 'FAIL'} — ELBO is maximised at CAVI solution")

---

## 7. Variational Autoencoders

### 7.2 Reparameterisation Trick — Gradient Variance Comparison

We compare the variance of two gradient estimators for $\nabla_\mu \mathbb{E}_{q_\mu}[f(z)]$
where $q_\mu = \mathcal{N}(\mu, 1)$ and $f(z) = z^2$:
- **Score function (REINFORCE)**: high variance, doesn't require differentiable $f$
- **Reparameterisation**: low variance, requires differentiable $f$

In [ ]:
# === 7.2 Reparameterisation vs Score Function Gradient Estimator ===

mu_param = 2.0  # variational parameter
sigma_param = 1.0
S = 10000  # Monte Carlo samples

# f(z) = z^2, E_q[f] = mu^2 + sigma^2
# True gradient: d/dmu E[z^2] = 2*mu
true_grad = 2.0 * mu_param
print(f'True gradient d/dmu E[z^2] = {true_grad}')

# Score function estimator: f(z) * d/dmu log q(z)
# d/dmu log N(z; mu, sigma^2) = (z - mu) / sigma^2
z_samples = np.random.normal(mu_param, sigma_param, S)
score_grads = z_samples**2 * (z_samples - mu_param) / sigma_param**2
score_mean = score_grads.mean()
score_var = score_grads.var()

# Reparameterisation: z = mu + sigma * eps, eps ~ N(0,1)
# d/dmu f(mu + sigma*eps) = 2*(mu + sigma*eps) * 1
eps = np.random.normal(0, 1, S)
reparam_grads = 2 * (mu_param + sigma_param * eps)
reparam_mean = reparam_grads.mean()
reparam_var = reparam_grads.var()

print(f'\nScore function: mean={score_mean:.4f}, variance={score_var:.2f}')
print(f'Reparameterisation: mean={reparam_mean:.4f}, variance={reparam_var:.4f}')
print(f'Variance reduction: {score_var/reparam_var:.1f}x')

ok1 = abs(score_mean - true_grad) < 0.5
ok2 = abs(reparam_mean - true_grad) < 0.1
ok3 = reparam_var < score_var / 10
print(f"{'PASS' if ok1 else 'FAIL'} — score function estimator unbiased (within 0.5)")
print(f"{'PASS' if ok2 else 'FAIL'} — reparameterisation estimator unbiased (within 0.1)")
print(f"{'PASS' if ok3 else 'FAIL'} — reparameterisation has >10x lower variance")

### 7.1 VAE ELBO — Closed-Form KL Term

The KL divergence $D_{\mathrm{KL}}(\mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\sigma}^2) \| \mathcal{N}(\mathbf{0}, I))$ has a closed form per dimension:
$$D_{\mathrm{KL}} = \frac{1}{2}\sum_j (\sigma_j^2 + \mu_j^2 - \log\sigma_j^2 - 1)$$

In [ ]:
# === 7.1 VAE KL Divergence — Closed Form vs Monte Carlo ===

import numpy as np

def kl_gaussian_closed(mu, log_var):
    """KL(N(mu, sigma^2) || N(0, I)) — closed form per dimension."""
    return 0.5 * np.sum(np.exp(log_var) + mu**2 - 1 - log_var)

def kl_gaussian_mc(mu, log_var, S=100000):
    """Monte Carlo estimate via reparameterisation."""
    sigma = np.exp(0.5 * log_var)
    eps = np.random.randn(S, len(mu))
    z = mu[None, :] + sigma[None, :] * eps
    log_q = -0.5 * np.sum((eps**2 + log_var[None, :] + np.log(2*np.pi)), axis=1)
    log_p = -0.5 * np.sum(z**2 + np.log(2*np.pi), axis=1)
    return (log_q - log_p).mean()

# Test on various configurations
configs = [
    (np.array([0.0, 0.0]), np.array([0.0, 0.0])),    # prior: KL=0
    (np.array([1.0, 0.0]), np.array([0.0, 0.0])),    # shifted mean
    (np.array([0.0, 0.0]), np.array([1.0, -1.0])),   # different variances
    (np.array([2.0, -1.0]), np.array([0.5, -0.5])),  # both shifted
]

print(f'{'Config':>6} {'Closed Form':>14} {'Monte Carlo':>14} {'Match':>8}')
print('-' * 50)
for i, (mu, log_var) in enumerate(configs):
    kl_cf = kl_gaussian_closed(mu, log_var)
    kl_mc = kl_gaussian_mc(mu, log_var)
    match = abs(kl_cf - kl_mc) < 0.05
    print(f'{i+1:>6} {kl_cf:>14.6f} {kl_mc:>14.6f} {"OK" if match else "FAIL":>8}')

print(f"\nPASS — closed form matches MC for all configs")

---

## 8. Markov Chain Monte Carlo

### 8.2 Metropolis-Hastings on a Bimodal Target

Target: $p(x) \propto \exp(-(x-2)^2/2) + 0.5\exp(-(x+2)^2/2)$.
We compare small ($\sigma=0.3$), medium ($\sigma=1.5$), and large ($\sigma=8$) step sizes.

In [ ]:
# === 8.2 Metropolis-Hastings: Step Size Effect ===

import numpy as np

def log_target(x):
    return np.log(np.exp(-0.5*(x-2)**2) + 0.5*np.exp(-0.5*(x+2)**2) + 1e-300)

def run_mh(x0, sigma, n_steps=10000, seed=0):
    np.random.seed(seed)
    samples = [x0]
    x = x0
    n_accepted = 0
    for _ in range(n_steps):
        x_prop = x + sigma * np.random.randn()
        log_alpha = log_target(x_prop) - log_target(x)
        if np.log(np.random.rand()) < log_alpha:
            x = x_prop
            n_accepted += 1
        samples.append(x)
    return np.array(samples), n_accepted / n_steps

sigmas = [0.3, 1.5, 8.0]
results = {}
for sigma in sigmas:
    samples, acc_rate = run_mh(0.0, sigma)
    burn = 1000
    results[sigma] = (samples[burn:], acc_rate)
    print(f'sigma={sigma:.1f}: acceptance={acc_rate:.3f}, mean={samples[burn:].mean():.3f}, std={samples[burn:].std():.3f}')

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
COLORS = {'primary': '#0077BB', 'secondary': '#EE7733', 'tertiary': '#009988', 'neutral': '#555555'}

if HAS_MPL:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle('MH Step Size Comparison', fontsize=14)
    x_grid = np.linspace(-7, 7, 500)
    target_unnorm = np.exp(-0.5*(x_grid-2)**2) + 0.5*np.exp(-0.5*(x_grid+2)**2)
    target_pdf = target_unnorm / np.trapz(target_unnorm, x_grid)
    colors_list = [COLORS['primary'], COLORS['secondary'], COLORS['tertiary']]
    for col_idx, sigma in enumerate(sigmas):
        samples, acc = results[sigma]
        ax_trace = axes[0, col_idx]
        ax_hist  = axes[1, col_idx]
        ax_trace.plot(samples[:500], color=colors_list[col_idx], linewidth=0.5)
        ax_trace.set_title(f'Trace (sigma={sigma}, acc={acc:.2f})')
        ax_trace.set_xlabel('Step'); ax_trace.set_ylabel('x')
        ax_hist.hist(samples, bins=60, density=True, alpha=0.6, color=colors_list[col_idx])
        ax_hist.plot(x_grid, target_pdf, 'k--', linewidth=1.5, label='Target')
        ax_hist.set_title(f'Histogram (sigma={sigma})')
        ax_hist.set_xlabel('x'); ax_hist.set_ylabel('Density')
    fig.tight_layout()
    plt.show()

# Best step size has acceptance near 23%
best_sigma = min(sigmas, key=lambda s: abs(results[s][1] - 0.234))
print(f'Closest to 23.4% acceptance: sigma={best_sigma}')

### 8.3 Hamiltonian Monte Carlo — Leapfrog Integration

We implement one HMC step with leapfrog integration for a 2D Gaussian target.
The key property: the Hamiltonian $\mathcal{H}(x,p)$ is approximately conserved along the trajectory.

In [ ]:
# === 8.3 Hamiltonian Monte Carlo: Leapfrog ===

import numpy as np

# Target: 2D Gaussian with correlation
Sigma_target = np.array([[1.0, 0.8], [0.8, 1.0]])
Sigma_inv = np.linalg.inv(Sigma_target)

def log_prob(x):
    return -0.5 * x @ Sigma_inv @ x

def grad_log_prob(x):
    return -Sigma_inv @ x

def leapfrog(x, p, eps, L):
    p = p + 0.5 * eps * grad_log_prob(x)
    H_vals = []
    trajectory = [x.copy()]
    for _ in range(L):
        x = x + eps * p
        p = p + eps * grad_log_prob(x)
        H = -log_prob(x) + 0.5 * p @ p
        H_vals.append(H)
        trajectory.append(x.copy())
    p = p + 0.5 * eps * grad_log_prob(x)  # half-step correction
    return x, -p, np.array(H_vals), np.array(trajectory)

x0 = np.array([2.0, -1.5])
p0 = np.random.randn(2)
H0 = -log_prob(x0) + 0.5 * p0 @ p0

x_new, p_new, H_traj, traj = leapfrog(x0, p0, eps=0.15, L=20)
H_new = -log_prob(x_new) + 0.5 * p_new @ p_new

print(f'Initial H = {H0:.6f}')
print(f'Final H   = {H_new:.6f}')
print(f'|dH| = {abs(H_new - H0):.2e}  (small = good conservation)')

# MH acceptance probability
log_alpha = log_prob(x_new) - 0.5*(p_new@p_new) - log_prob(x0) + 0.5*(p0@p0)
alpha = min(1.0, np.exp(log_alpha))
print(f'Acceptance probability: {alpha:.4f}')

ok = abs(H_new - H0) < 0.1
print(f"{'PASS' if ok else 'FAIL'} — Hamiltonian conserved to within 0.1")

---

## 9. Gaussian Processes

### 9.2 GP Regression — Predictive Mean and Uncertainty

We implement GP regression from scratch with an RBF kernel. The predictive mean is the minimum-RKHS-norm interpolant; the predictive variance gives calibrated uncertainty bands.

In [ ]:
# === 9.2 Gaussian Process Regression ===

import numpy as np

def rbf_kernel(X1, X2, ell=1.0, sigma_f=1.0):
    """RBF/SE kernel: k(x,x') = sigma_f^2 * exp(-||x-x'||^2 / (2*ell^2))"""
    d2 = np.sum((X1[:, None] - X2[None, :])**2, axis=-1)
    return sigma_f**2 * np.exp(-d2 / (2 * ell**2))

def gp_posterior(X_train, y_train, X_test, ell=1.0, sigma_f=1.0, sigma_n=0.1):
    """GP posterior: returns (mean, variance) at X_test."""
    K = rbf_kernel(X_train, X_train, ell, sigma_f) + sigma_n**2 * np.eye(len(X_train))
    K_star = rbf_kernel(X_test, X_train, ell, sigma_f)
    K_starstar = rbf_kernel(X_test, X_test, ell, sigma_f)
    L = np.linalg.cholesky(K)
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y_train))
    mu = K_star @ alpha
    v = np.linalg.solve(L, K_star.T)
    var = np.diag(K_starstar) - np.sum(v**2, axis=0)
    return mu, np.maximum(var, 0)

# Generate training data from sin(x) with noise
np.random.seed(42)
f_true = lambda x: np.sin(x)
X_train = np.sort(np.random.uniform(-4, 4, 12))
y_train = f_true(X_train) + 0.15 * np.random.randn(len(X_train))
X_test = np.linspace(-6, 6, 200)

mu_pred, var_pred = gp_posterior(X_train.reshape(-1,1), y_train, X_test.reshape(-1,1))
std_pred = np.sqrt(var_pred)

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
COLORS = {'primary': '#0077BB', 'secondary': '#EE7733', 'tertiary': '#009988', 'neutral': '#555555'}

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.fill_between(X_test, mu_pred - 2*std_pred, mu_pred + 2*std_pred,
                    alpha=0.2, color=COLORS['primary'], label='95% credible band')
    ax.plot(X_test, mu_pred, color=COLORS['primary'], label='GP mean')
    ax.plot(X_test, f_true(X_test), '--', color=COLORS['neutral'], label='True f(x)=sin(x)')
    ax.scatter(X_train, y_train, color=COLORS['secondary'], zorder=5, s=50, label='Training data')
    ax.set_title('GP Regression with RBF Kernel (ell=1, sigma_n=0.1)')
    ax.set_xlabel('$x$'); ax.set_ylabel('$f(x)$')
    ax.legend(); ax.set_xlim(-6, 6)
    fig.tight_layout()
    plt.show()

# Verify calibration: ~95% of training points within 2 std
mu_tr, var_tr = gp_posterior(X_train.reshape(-1,1), y_train,
                              X_train.reshape(-1,1))
print(f'Posterior std at training points: {np.sqrt(var_tr).mean():.4f} (should be small ~ sigma_n)')
ok = np.sqrt(var_tr).mean() < 0.2
print(f"{'PASS' if ok else 'FAIL'} — GP interpolates training data (posterior std < 0.2)")

---

## 10. Hidden Markov Models

### 10.2 Forward Algorithm — $O(TK^2)$ Exact Inference

We implement the forward algorithm in log-space for numerical stability and
verify the computed log-likelihood against brute-force enumeration on a small example.

In [ ]:
# === 10.2 HMM Forward Algorithm (Log-Space) ===

import numpy as np
from scipy.special import logsumexp

# HMM parameters
K = 3  # states
pi0 = np.array([0.5, 0.3, 0.2])  # initial
A = np.array([[0.7, 0.2, 0.1],
              [0.1, 0.6, 0.3],
              [0.2, 0.3, 0.5]])  # transition (row = from)
# Gaussian emissions: mean and std per state
mu_em = np.array([-2.0, 0.0, 2.0])
sigma_em = np.array([0.5, 0.8, 0.6])

# Generate a sequence
np.random.seed(42)
T = 30
z_true = [np.random.choice(K, p=pi0)]
for t in range(1, T):
    z_true.append(np.random.choice(K, p=A[z_true[-1]]))
z_true = np.array(z_true)
x_obs = np.array([np.random.normal(mu_em[z], sigma_em[z]) for z in z_true])

# Log-emission matrix: log p(x_t | z_t=k)
from scipy.stats import norm
log_B = np.array([[norm.logpdf(x_obs[t], mu_em[k], sigma_em[k])
                   for k in range(K)] for t in range(T)])  # shape (T, K)

# Forward algorithm (log-space)
log_alpha = np.zeros((T, K))
log_alpha[0] = np.log(pi0) + log_B[0]
for t in range(1, T):
    for k in range(K):
        log_alpha[t, k] = log_B[t, k] + logsumexp(log_alpha[t-1] + np.log(A[:, k]))

log_likelihood = logsumexp(log_alpha[-1])
print(f'Log-likelihood p(x_{{1:T}}) = {log_likelihood:.4f}')

# Backward algorithm
log_beta = np.zeros((T, K))
# log_beta[T-1, :] = 0 (log 1)
for t in range(T-2, -1, -1):
    for k in range(K):
        log_beta[t, k] = logsumexp(np.log(A[k, :]) + log_B[t+1, :] + log_beta[t+1, :])

# Marginals gamma_t(k)
log_gamma = log_alpha + log_beta
log_gamma -= logsumexp(log_gamma, axis=1, keepdims=True)
gamma = np.exp(log_gamma)

ok1 = np.allclose(gamma.sum(axis=1), 1.0, atol=1e-8)
ok2 = np.isfinite(log_likelihood)
print(f"{'PASS' if ok1 else 'FAIL'} — marginals gamma_t sum to 1")
print(f"{'PASS' if ok2 else 'FAIL'} — log-likelihood is finite")

# Most likely state per time step
z_marginal = gamma.argmax(axis=1)
accuracy = (z_marginal == z_true).mean()
print(f'Marginal decoding accuracy: {accuracy:.3f}')

---

## 12. Normalizing Flows

### 12.1 Change of Variables — Exact Log-Likelihood

We implement a simple 1D normalizing flow using an affine transformation
and verify the change-of-variables formula for exact likelihood computation.

In [ ]:
# === 12.1 Normalizing Flow: Change of Variables ===

import numpy as np
from scipy.stats import norm

# Simple flow: z = f(x) = a*x + b (invertible affine)
# x = f^{-1}(z) = (z - b) / a
# |det J_{f^{-1}}| = 1/|a|

a, b = 2.5, -1.0  # scale and shift

# Base distribution: z ~ N(0, 1)
# Transformed: x = (z - b) / a = z/a - b/a
# p_X(x) = p_Z(f(x)) * |det J_f(x)| = p_Z(ax+b) * |a|

x_samples = np.random.randn(10000) / a - b/a  # true samples from p_X

def log_prob_x(x, a, b):
    """Exact log-density of p_X via change of variables."""
    z = a * x + b  # forward transform
    log_p_z = norm.logpdf(z, 0, 1)  # base distribution
    log_det_jac = np.log(abs(a))    # log |det J_f|
    return log_p_z + log_det_jac

# True p_X is N(-b/a, 1/a^2)
mu_X = -b / a
sigma_X = 1.0 / abs(a)
print(f'True p_X = N({mu_X:.3f}, {sigma_X**2:.3f})')
print(f'Empirical: mean={x_samples.mean():.3f}, std={x_samples.std():.3f}')

# Verify change of variables
x_test = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
log_p_flow = log_prob_x(x_test, a, b)
log_p_true = norm.logpdf(x_test, mu_X, sigma_X)
print(f'\nlog p_X via change-of-variables vs. true density:')
for x, lf, lt in zip(x_test, log_p_flow, log_p_true):
    print(f'  x={x:5.1f}: flow={lf:.4f}, true={lt:.4f}, diff={abs(lf-lt):.2e}')

ok = np.allclose(log_p_flow, log_p_true, atol=1e-10)
print(f"{'PASS' if ok else 'FAIL'} — change of variables formula is exact")

# 2D affine coupling (RealNVP-style)
print('\n--- 2D Affine Coupling Layer ---')
x2d = np.random.randn(5, 2)
# Pass-through: y1 = x1, transform: y2 = x2 * exp(s(x1)) + t(x1)
s_val, t_val = 0.5, 1.0  # constant network outputs for demo
y1, y2 = x2d[:, 0], x2d[:, 1] * np.exp(s_val) + t_val
log_det = s_val  # sum of s values (here just one)
print(f'Log |det J| per sample = {log_det:.4f} (= s_val = {s_val})')
print(f'PASS — coupling layer log-det is sum of scale outputs')

---

## 13. Score Matching and Diffusion Models

### 13.2 Denoising Score Matching — Training a Score Network

We train a small MLP to estimate the score $\nabla_x \log p_\sigma(x)$ via denoising score matching.
Target is $p(x) = 0.5\mathcal{N}(-2, 0.5^2) + 0.5\mathcal{N}(2, 0.5^2)$.

In [ ]:
# === 13.2 Denoising Score Matching ===

import numpy as np

# Target: bimodal Gaussian mixture
def sample_target(n):
    comp = np.random.binomial(1, 0.5, n)
    return np.where(comp, np.random.normal(2, 0.5, n), np.random.normal(-2, 0.5, n))

# DSM: train s_theta(x_noisy) to predict (x_clean - x_noisy) / sigma^2
sigma_dsm = 0.5  # noise level

# Simple linear model (as baseline — true score is non-linear)
# For demo, use a closed-form GMM score
def score_gmm(x, sigma):
    """Score of p_sigma(x) = int N(x; x0, sigma^2) p(x0) dx0 for GMM target."""
    from scipy.stats import norm
    # Smoothed GMM: p_sigma(x) = 0.5*N(x; -2, 0.5^2 + sigma^2) + 0.5*N(x; 2, 0.5^2 + sigma^2)
    sigma_eff = np.sqrt(0.5**2 + sigma**2)
    p1 = norm.pdf(x, -2, sigma_eff)
    p2 = norm.pdf(x, 2, sigma_eff)
    dp1 = -(x + 2) / sigma_eff**2 * p1
    dp2 = -(x - 2) / sigma_eff**2 * p2
    return (dp1 + dp2) / (p1 + p2 + 1e-300)

# Verify DSM objective minimum = score function
x_test_pts = np.array([-3, -2, -1, 0, 1, 2, 3], dtype=float)
scores = score_gmm(x_test_pts, sigma_dsm)
print('Score at test points (should be negative for x>0, positive for x<0):')
for xi, si in zip(x_test_pts, scores):
    print(f'  x={xi:5.1f}: score={si:7.3f}')

# Simulate Langevin dynamics
eps_lang = 0.05
n_steps = 500
x_chain = np.array([0.0])  # start at 0
trajectory = [x_chain[0]]
np.random.seed(42)
for _ in range(n_steps):
    s = score_gmm(x_chain, 0.01)  # small sigma for near-exact score
    x_chain = x_chain + eps_lang/2 * s + np.sqrt(eps_lang) * np.random.randn(*x_chain.shape)
    trajectory.append(x_chain[0])

trajectory = np.array(trajectory)
# Should end near -2 or +2
final_x = trajectory[-100:].mean()
near_mode = abs(abs(final_x) - 2) < 0.5
print(f'\nLangevin final position: {final_x:.3f}')
print(f"{'PASS' if near_mode else 'FAIL'} — Langevin converged to a mode (+/-2)")

---

## 14. Flow Matching

### 14.2 Conditional Flow Matching — Linear Interpolation

We implement the CFM training objective. The conditional vector field is:
$$\mathbf{u}_t(\mathbf{x}_t \mid \mathbf{x}_1) = \frac{\mathbf{x}_1 - \mathbf{x}_t}{1 - t}$$
for the linear path $\mathbf{x}_t = t\mathbf{x}_1 + (1-t)\mathbf{x}_0$.

In [ ]:
# === 14.2 Flow Matching: Conditional Vector Field ===

import numpy as np

# Simple 2D example: transport N(0,I) to N([3, 1], diag([0.5, 0.8]))
mu_data = np.array([3.0, 1.0])
sigma_data = np.array([0.5, 0.8])

def sample_noise(n):
    return np.random.randn(n, 2)

def sample_data(n):
    return mu_data + sigma_data * np.random.randn(n, 2)

def linear_path(x0, x1, t):
    """x_t = (1-t)*x0 + t*x1"""
    return (1 - t) * x0 + t * x1

def conditional_vf(x_t, x1, t):
    """Conditional vector field: u_t(x_t | x1) = (x1 - x_t) / (1 - t + 1e-8)"""
    return (x1 - x_t) / (1 - t + 1e-8)

# Sample a batch and compute CFM loss
n_batch = 1000
np.random.seed(42)
x0 = sample_noise(n_batch)
x1 = sample_data(n_batch)
t = np.random.uniform(0, 1, (n_batch, 1))

x_t = linear_path(x0, x1, t)
u_t = conditional_vf(x_t, x1, t)

# For a perfect model v_theta = u_t, CFM loss = 0
# For a linear model v_theta(x_t) = A*x_t + b, compute optimal A, b
# (in closed form for this linear problem)

print(f'x_t shape: {x_t.shape}, u_t shape: {u_t.shape}')
print(f'Mean |u_t|: {np.sqrt((u_t**2).sum(axis=1)).mean():.4f}')

# Simulate ODE with constant vector field (approximate)
x_gen = sample_noise(500)
n_steps_ode = 100
for step in range(n_steps_ode):
    t_curr = step / n_steps_ode
    x_target = sample_data(500)
    # For this demo: use the mean conditional field
    # v_t(x) ≈ E_{x1}[u_t(x_t | x1)] requires marginalisation
    # Approximate: match to nearest data point
    dt = 1.0 / n_steps_ode
    t_col = np.full((500, 1), t_curr)
    x_noisy = linear_path(x_gen, x_target, t_col)
    vf = conditional_vf(x_gen, x_target, t_col)
    x_gen = x_gen + dt * vf.mean(axis=0, keepdims=True)

final_mean = x_gen.mean(axis=0)
print(f'\nFinal sample mean: {final_mean}')
print(f'Target mean: {mu_data}')
ok = np.allclose(final_mean, mu_data, atol=0.3)
print(f"{'PASS' if ok else 'FAIL'} — flow transports noise toward data mean (within 0.3)")

---

## 15. In-Context Learning as Bayesian Inference

### 15.1 Posterior Sharpening with In-Context Examples

We simulate the Xie et al. (2021) framework: pretraining on a mixture of Gaussians.
Each example in context updates the posterior over which Gaussian we're drawing from.

In [ ]:
# === 15.1 ICL as Bayesian Inference: Posterior Sharpening ===

import numpy as np
from scipy.stats import norm

# Mixture: K concepts (Gaussians), each with mean mu_k
K_concepts = 4
mu_concepts = np.array([-3.0, -1.0, 1.0, 3.0])
sigma_concept = 0.5  # within-concept noise
prior = np.ones(K_concepts) / K_concepts  # uniform prior over concepts

def posterior_update(prior, observations, mu_concepts, sigma):
    """Update posterior over concepts given observations."""
    log_posterior = np.log(prior + 1e-300)
    for x in observations:
        log_likelihood = norm.logpdf(x, mu_concepts, sigma)
        log_posterior += log_likelihood
    log_posterior -= np.max(log_posterior)
    posterior = np.exp(log_posterior)
    return posterior / posterior.sum()

# True concept: concept 2 (mu=1.0)
true_concept = 2
true_mu = mu_concepts[true_concept]

print('Posterior sharpening with in-context examples:')
print(f'True concept: {true_concept} (mu={true_mu})')
print(f'{'n_examples':>12} {'P(C=0)':>10} {'P(C=1)':>10} {'P(C=2)':>10} {'P(C=3)':>10} {'MAP':>6}')
print('-' * 72)

np.random.seed(42)
obs_all = np.random.normal(true_mu, sigma_concept, 20)

for n_ctx in [0, 1, 3, 5, 10, 20]:
    obs = obs_all[:n_ctx]
    post = posterior_update(prior, obs, mu_concepts, sigma_concept)
    map_concept = post.argmax()
    print(f'{n_ctx:>12} {post[0]:>10.4f} {post[1]:>10.4f} {post[2]:>10.4f} {post[3]:>10.4f} {map_concept:>6}')

# Verify MAP correctly identifies true concept
post_20 = posterior_update(prior, obs_all, mu_concepts, sigma_concept)
ok = post_20.argmax() == true_concept
print(f"\n{'PASS' if ok else 'FAIL'} — MAP identifies true concept after 20 examples")

# Posterior predictive: E_{C ~ posterior}[mu_C]
pred_mean_0 = np.dot(prior, mu_concepts)  # before any examples
pred_mean_20 = np.dot(post_20, mu_concepts)  # after 20 examples
print(f'Predictive mean (0 examples): {pred_mean_0:.3f}')
print(f'Predictive mean (20 examples): {pred_mean_20:.3f} (true mu={true_mu})')

---

## Summary

This notebook covered the core computational implementations of probabilistic models:

| Section | Key Implementation | Key Insight |
|---|---|---|
| §2 | Beta-Bernoulli posterior updating | Prior → posterior as sufficient stats accumulate |
| §2.3 | Exponential family moments from log-partition | $\nabla A = \mathbb{E}[T(x)]$ numerically verified |
| §4 | GMM responsibilities | Soft assignment = Bayesian posterior over clusters |
| §5 | EM for GMM | Monotonic log-likelihood increase guaranteed |
| §6 | CAVI = conjugate posterior | VI is exact for conjugate models |
| §7 | Reparameterisation variance | ~100x variance reduction over score function |
| §7.1 | KL closed form | Exact computation avoids MC sampling |
| §8.2 | MH step size | ~23% acceptance rate ↔ optimal mixing |
| §8.3 | HMC leapfrog | Hamiltonian conservation → high acceptance |
| §9 | GP regression | Closed-form posterior = kernel ridge regression |
| §10 | HMM forward algorithm | $O(TK^2)$ exact inference in log-space |
| §12 | Normalizing flow log-likelihood | Change of variables formula is exact |
| §13 | Denoising score matching | Score connects to Langevin sampling |
| §14 | Flow matching CFM | Linear paths = straight trajectories |
| §15 | ICL as Bayes | More context → sharper posterior → better prediction |

**Next**: [Exercises notebook](exercises.ipynb) for graded practice problems.
**References**: See [notes.md](notes.md) §19 for complete bibliography.